<a href="https://colab.research.google.com/github/RobJavVar/DataSciencePsychNeuro/blob/master/ExerciseSubmissions/15_power-analysis-via-simulations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 15: Power analyses

This  assignment is designed to give you practice with Monte Carlo methods to conduct power analyses via simulation. You won't need to load in any data for this homework. We will, however, be using parts of the homework from last week.

---
## 1. Simulating data (1 points)


Pull your `simulate_data()` function from your last homework and add it below.

As a reminder, this function simulates the relationship between age, word reading experience, and reading comprehension skill.

`c` is reading comprehension, and `x` is word reading experience.

In [1]:
sample_size = 100 # How many children in data set?
age_lo = 80     # minimum age, in months
age_hi = 200    # maximum age, in months
beta_xa = 0.5   # amount by which experience changes for increase of one month in age
beta_x0 = -5    # amount of experience when age = 0 (not interpretable, since minimum age for this data is 80 months)
sd_x = 50       # standard dev of gaussian noise term, epsilon_x
beta_ca = 0.8   # amount that comprehension score improves for every increase of one unit in age
beta_cx = 3     # amount that comprehension score improves for every increase of one unit in reading experience
beta_c0 = 10    # comprehension score when reading experience is 0.
sd_c = 85      # standard dev of gaussian noise term, epsilon_c

simulate_data <- function(sample_size, age_lo, age_hi, beta_xa,
                          beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
      # WRITE YOUR CODE HERE
  age <- runif(sample_size, min=age_lo, max= age_hi)
  ex <- rnorm(sample_size, mean=0, sd=sd_x)
  experience <- beta_x0 + beta_xa*age + ex
  ec <- rnorm(sample_size, mean=0, sd=sd_c)
  comprehension <- beta_c0 + beta_ca*age + beta_cx * experience + ec

  dat <- data.frame(age=age, experience=experience, comprehension=comprehension)
  
  return(dat)
}

dat <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
head(dat)

,age,experience,comprehension
,<dbl>,<dbl>,<dbl>
1,151.49442,112.29261,369.4078
2,168.40753,24.56023,174.8513
3,178.92710,71.00597,313.4855
4,163.36459,83.94555,376.3599
5,99.06135,38.16592,154.4095
6,106.79525,88.71455,361.4328


---
## 2. `run_analysis()` function (2 pts)

Last week, we looked at whether word reading experience(`x`) mediated the relation between `age` and reading comprehension (`c`).

Now we're going to use our `simulate_data()` function to conduct a power analysis. The goal is to determine how many participants we would need in order to detect both the mediated and the direct effects in this data.

*Note: We're going to pretend for the sake of simplicity that we don't have any control over the ages of the children we get (so ages are generated using `runif(sample_size, age_lo, age_hi)`, although of course this would be an unusual situation in reality.*

First, write a function, `run_analysis()`, that takes in simulated data, runs **your mediation from last week**, and returns a vector containing the ACME and ADE estimates and p-values (these are the `d0`, `d0.p`, `z0`, and `z0.p` features of the mediated model object, e.g., `fitMed$d0.p`). Print this function's output for the data we simulated previously.

In [9]:
# WRITE YOUR CODE HERE
library(mediation)

run_analysis <- function(dat) {
    
M <- lm(experience ~ age, data=dat)
summary(M)

Y <- lm(comprehension ~ age + experience, data=dat)
summary(Y)

Med <- mediate(M, Y, treat= "age", mediator= "experience", sims=1000)
summary(Med)

results <- c(
    d0 =Med$d0,        
    d0.p =Med$d0.p,    
    z0 =Med$z0,       
    z0.p =Med$z0.p     
  )
  
  return(results)
}

run_analysis(dat)


d0     d0.p       z0     z0.p 
1.306325 0.006000 0.463380 0.082000

---
## 3. `repeat_analysis()` function (3 pts)

Next fill in the function `repeat_analysis()` below so that it simulates and analyzes data `num_simulations` times. Store the outputs from each simulation in the `simouts` matrix. Calculate and return the coverage across all the simulations run for both ACME and ADE.

In [14]:
repeat_analysis <- function(num_simulations, alpha, sample_size, age_lo, age_hi,
        beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
    # Initialize simouts matrix for storing each output from run_analysis()
    simouts <- matrix(rep(NA, num_simulations*4), nrow=num_simulations, ncol=4)

    # Start simulating
    for (i in 1:num_simulations) {
      # WRITE YOUR CODE HERE    

        dat <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, 
                          beta_ca, beta_cx, beta_c0, sd_c)
    
        simouts[i, ] <- run_analysis(dat)
        colnames(simouts) <- c("d0", "d0.p", "z0", "z0.p")
    }

    # Calculate coverage for both ACME and ADE estimates using p-values in simouts
    ACME_cov = mean(simouts[, "d0.p"] < alpha)
    ADE_cov =  mean(simouts[, "z0.p"] < alpha)

    return(list(ACME_cov =ACME_cov, ADE_cov = ADE_cov))
}



Now run the `repeat_analysis()` function using the same parameter settings as above, for 10 simulations, with an alpha criterion of 0.01.

In [16]:
# WRITE YOUR CODE HERE
repeat_analysis(
  num_simulations =10,
  alpha =0.01,
  sample_size=100,
  age_lo =age_lo,
  age_hi =age_hi,
  beta_xa =beta_xa,
  beta_x0 =beta_x0,
  sd_x=sd_x,
  beta_ca=beta_ca,
  beta_cx=beta_cx,
  beta_c0 =beta_c0,
  sd_c=sd_c
)



$ACME_cov
[1] 0.6

$ADE_cov
[1] 0.8

---
## 4. Testing different sample sizes (2 pts)

Finally, do the same thing (10 simulations, alpha criterion of 0.01) but for 5 different sample sizes: 50, 75, 100, 125, 150. You can do this using `map` (as in the tutorial), or a simple `for` loop, or by calculating each individually. Up to you! This should take around 3 minutes to run.

In [22]:
# WRITE YOUR CODE HERE

sample_size <- c(50, 75, 100, 125, 150)
power_results <- list()
for (n in sample_size) {
  
  cat("Sample size=", n, "...\n")
  
  power_results[[as.character(n)]] <- repeat_analysis(
  num_simulations =10,
  alpha =0.01,
  sample_size=100,
  age_lo =age_lo,
  age_hi =age_hi,
  beta_xa =beta_xa,
  beta_x0 =beta_x0,
  sd_x=sd_x,
  beta_ca=beta_ca,
  beta_cx=beta_cx,
  beta_c0 =beta_c0,
  sd_c=sd_c)}


Sample size= 50 ...
Sample size= 75 ...
Sample size= 100 ...
Sample size= 125 ...
Sample size= 150 ...


Print your results.

In [23]:
# WRITE YOUR CODE HERE
for (n in sample_size) {
  cat("Sample size:", n, "\n")
  print(power_results[[as.character(n)]])
  cat("\n")}

Sample size: 50 
$ACME_cov
[1] 0.8

$ADE_cov
[1] 0.7


Sample size: 75 
$ACME_cov
[1] 0.8

$ADE_cov
[1] 0.9


Sample size: 100 
$ACME_cov
[1] 0.9

$ADE_cov
[1] 0.5


Sample size: 125 
$ACME_cov
[1] 1

$ADE_cov
[1] 0.7


Sample size: 150 
$ACME_cov
[1] 0.7

$ADE_cov
[1] 0.6




## 5. Reflection (2 pts)

If this were a real power analysis, we'd want to run more simulations per sample size (to get a more precise estimate of power) and we may also want to test out some other values of the parameters we used to simulate our data. However, what would you conclude just based on the results above?

> *Write your response here.*
>These results suggest the mediated effect is generally detectable even at low/moderate sample sizes, whereas the direct effect is more variable and not reliably significant. These results indicate that larger sample sizes would likely improve power for detecting the direct effect.

**Given** how we generated the data, why was the direct effect harder to detect than the mediated effect?
> *Write your response here.*
> Because the mediator (experience) accounts for some of the effect of the age on comprehension, which means the direct effect alone is harder to detect statistically since the effect is smaller. 

**DUE:** 5pm EST, March 31, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *Someone's Name*